In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import numpy as np

load_dotenv()
client = OpenAI()  # reads OPENAI_API_KEY from env

In [2]:
def embed(text: str) -> list[float]:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text,
    )
    return np.array(response.data[0].embedding)

In [3]:
vector = embed("hello world")
print(f"dimensions: {len(vector)}")
print(f"first 5 values: {vector[:5]}")

dimensions: 1536
first 5 values: [-0.00679398 -0.03918457  0.03411865  0.0287323  -0.02481079]


In [4]:

def similarity(a: list[float], b: list[float]) -> float:
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))

# try it
w1 = "capital"
w2 = "paris"
w3 = "france"

v1 = embed(w1)
v2 = embed(w2)
v3 = embed(w3)
#v3 = embed("spaceship")

print(f"{w1} vs {w2}:       {similarity(v1, v2):.4f}")
print(f"{w1} vs {w2} - {w3}:       {similarity(v1, (v2-v3)):.4f}")
print(f"{w2} vs {w3}:       {similarity(v2, v3):.4f}")
print(f"{w3} vs {w2} - {w1}:       {similarity(v3, (v2-v1)):.4f}")

capital vs paris:       0.3071
capital vs paris - france:       0.1258
paris vs france:       0.6087
france vs paris - capital:       0.3506


## Span pooling with all-mpnet-base-v2

Extracts a contextual vector for a specific entity span by pooling only the token embeddings that overlap with that span's character offsets. Requires: `uv add transformers torch`

In [6]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL = "sentence-transformers/all-mpnet-base-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)
model.eval()
print("model loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model loaded


In [9]:
MAX_TOKENS = 512
CHARS_PER_TOKEN = 4  # rough estimate for windowing

def embed_span(text: str, span_start: int, span_end: int) -> list[float]:
    """Return a contextual vector for text[span_start:span_end] within text."""
    # if text is too long, extract a window around the span
    if len(text) > MAX_TOKENS * CHARS_PER_TOKEN:
        window = (MAX_TOKENS * CHARS_PER_TOKEN) // 2
        chunk_start = max(0, span_start - window)
        chunk_end = min(len(text), span_end + window)
        text = text[chunk_start:chunk_end]
        span_start -= chunk_start
        span_end -= chunk_start

    encoding = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=MAX_TOKENS,
    )
    offset_mapping = encoding.pop("offset_mapping")[0]

    with torch.no_grad():
        output = model(**encoding)

    token_embeddings = output.last_hidden_state[0]

    mask = torch.tensor([
        tok_start < span_end and tok_end > span_start
        for tok_start, tok_end in offset_mapping.tolist()
    ])

    span_tokens = token_embeddings[mask]
    if span_tokens.shape[0] == 0:
        span_tokens = token_embeddings

    return span_tokens.mean(dim=0).numpy().tolist()

In [10]:
def embed_sentence(text: str) -> list[float]:
    """Mean-pool all tokens to get a full sentence embedding."""
    encoding = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        output = model(**encoding)
    return output.last_hidden_state[0].mean(dim=0).numpy().tolist()


sentence1 = "Joe Smith is a publisher in LA working on sports content"
sentence2 = "Joe the computer science major in LA"
sentence3 = "Karl is the sports writer living in Los Angeles"

v1 = embed_sentence(sentence1)
v2 = embed_sentence(sentence2)
v3 = embed_sentence(sentence3)

print(f"sports publisher vs CS Joe:     {similarity(v1, v2):.4f}")
print(f"sports publisher vs sports Joe: {similarity(v1, v3):.4f}")
print(f"  — higher second score = model sees Joe Smith and sports Joe as more related")

sports publisher vs CS Joe:     0.5066
sports publisher vs sports Joe: 0.5231
  — higher second score = model sees Joe Smith and sports Joe as more related


In [11]:
# Long sentence with multiple entities
doc = "Joe Smith, a sports journalist from LA, interviewed Apple CEO Tim Cook about the new iPhone launch at SoFi Stadium."

entities = {
    "Joe Smith":    "person / sports journalist",
    "Apple":        "tech company",
    "Tim Cook":     "person / tech CEO",
    "iPhone":       "tech product",
    "SoFi Stadium": "sports venue / LA",
}

# extract span vector for each entity
vectors = {}
for entity in entities:
    start = doc.index(entity)
    vectors[entity] = embed_span(doc, start, start + len(entity))

# compare all pairs
print(f"{'Entity A':<14} {'Entity B':<14} {'Similarity':>10}  context")
print("-" * 60)
entity_list = list(entities.keys())
for i in range(len(entity_list)):
    for j in range(i + 1, len(entity_list)):
        a, b = entity_list[i], entity_list[j]
        score = similarity(vectors[a], vectors[b])
        print(f"{a:<14} {b:<14} {score:>10.4f}  ({entities[a]} | {entities[b]})")

Entity A       Entity B       Similarity  context
------------------------------------------------------------
Joe Smith      Apple              0.5197  (person / sports journalist | tech company)
Joe Smith      Tim Cook           0.6001  (person / sports journalist | person / tech CEO)
Joe Smith      iPhone             0.3319  (person / sports journalist | tech product)
Joe Smith      SoFi Stadium       0.4736  (person / sports journalist | sports venue / LA)
Apple          Tim Cook           0.7383  (tech company | person / tech CEO)
Apple          iPhone             0.6662  (tech company | tech product)
Apple          SoFi Stadium       0.4877  (tech company | sports venue / LA)
Tim Cook       iPhone             0.5353  (person / tech CEO | tech product)
Tim Cook       SoFi Stadium       0.5039  (person / tech CEO | sports venue / LA)
iPhone         SoFi Stadium       0.4899  (tech product | sports venue / LA)


In [12]:
from enum import Enum
from pydantic import BaseModel

class EntityType(str, Enum):
    PERSON = "PERSON"
    ROLE = "ROLE"
    ORG = "ORG"
    CONCEPT = "CONCEPT"
    TOOL = "TOOL"
    LOCATION = "LOCATION"
    EVENT = "EVENT"

class Entity(BaseModel):
    text: str
    type: EntityType

class Entities(BaseModel):
    entities: list[Entity]

def chunk_doc(text: str, max_chars: int = 1000) -> list[str]:
    """Split by double newline, merge small paragraphs into chunks."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) <= max_chars:
            current = (current + "\n\n" + para).strip()
        else:
            if current:
                chunks.append(current)
            current = para
    if current:
        chunks.append(current)
    return chunks

def extract_entities(chunk: str) -> list[Entity]:
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract all named entities. Return only exact strings as they appear in the text."},
            {"role": "user", "content": chunk},
        ],
        response_format=Entities,
    )
    return response.choices[0].message.parsed.entities

doc = """based on [[Breaking point for Buyers of AI KMS]] and [[Potential Markets]] clearly I need to reach out to 
## **Reach out to CTO's or COO's of a late Seed or Series A B2B SaaS startups**

- ==Late Seed / Series A startups==
	- raised seed couple months ago to feel the pain
	- Fintech, biotech, B2B SaaS
	- **Champions + Buyers**
		- **CTO** / VP of Engineering ==best at seed==
			- answers lots of questions
			- knows KM and its problems
			- owns the tools budget
			- under pressure from stakeholders to reduce per-person risk (KM problme )
		- **COO** / Head of Ops <-- LITERALLY OWNS THE PROBLEM ==best at Ser A==
			- **their job is to run company without founders touching everything**
			- slow onboarding, repeated questions, tribal knowledge <- bad!
			- will drive adoption, owns budget, has vast interest
		- **CEO** / co-founder
			- "why does everyone keep asking me the same questions."
			- May delegate
		- **Senior engineer** / tech lead <-- ==USER and advocate==
			- lives in the pain
			- **bottom-up demand signal** 

Extra Factors
- first painful departure -- lost info
- team crossing ~12 ppl
	- work duplication, repeated slack questions
- hiring of highly technical ppl who cannot find anything
- post-incident relief

![[Screenshot 2026-05-14 at 12.04.09.png]]


We talked for about an hour -> 
- 30 minutes on her problems
- 30 minutes on my solution

==Gold==
==Unknown Unknowns== -- ppl do not now what to even search for! This is where GraphRAG + agents beat traditional FAQ RAG
## Reflection
**Improvements to be done**
- Think about how to prepare for the product pitch
	- Think in advance what **FEATURES DO MATTER FOR THEM**
	- There is a ton of features I have, but I have time to pitch only few
		- As the competitor to the known solutions (eg. FAQ RAG)
		- As the general AI practices I implement opposing to what they know (eg. ambiguity resolution by asking follow-ups)
	- Think in advance what real **CASE STUDY MATTERS TO THEM**
		- like a security HR versus international HR 
			- ppl may think they need the international, but the tool will direct them to the security one
- Learn how to ask money related questions
- Be more bold in the next steps
	- At least what I can do is it mention it in the follow-up
- Warm up the contact between first and the last interactions
- IMPORTANT: write things down RIGHT AFTER the meet-up to avoid loosing the small details as I did today (hopefully not)
- Next time, ask about how much time she spends on it
- In the ideal world
	- the whoever I talked to would make this a one of the top priorities in order to solve the burning pain
	- will feel the Eureka moment when they derive the value of my product by naming vocalizing their problems
- Vocalizing their problems helps them to see the value in my product!

**What went well**
- Ppl love to talk about their problems! Especially when they see understanding and appreciation
	- No one listens to their problems, so clearly I bring a ton of value to them by hearing out and asking follow-ups!
- Smiling a lot! Ppl just become more positive!
- **Mom test nails it!**
- Provide value first! Like a kind salesman -most probably they don't have the expertise and time to build it anyway.
	- Give it as an option for the sake of options, so they choose your expertise, not just the blackbox product --> ==transparency and options availability builds trust!==
- I have explained how it works, so she may try to implement it by herself and most probably fail because there is a need for the background I have build up
- I think I did well in showing my knowledge about security measures like [[Zero Data Retention]]
- It was also nice to show the tool real-time answering her question right away
- It it rare to build things fast, so I managed to surprise her by agility I guess
- **Say that the tool is model / DB host agnostic and can be made over what company trusts already** -- because it is a hidden cost for them to adapt to a new system
- I explained the theory nicely on "why graphs?"
	- I explained her the standard RAG (would be nice to give a use case example like FAQ)
	- I explained her the property Graph and how seeding works
		- was nice to show that the search finds the neighbouring nodes to answer vague questions
	- Then she (maybe through my explanation) derived the benefit of using graph over the normal table <-- **THIS COULD BE A WINNING POINT -- PPL DERIVE VALUE BY THEMSELVES!!**
	- Giving a tangible case-study is really helpful
- **It is helpful to draw** <- Tess's testimony
- She liked cloud inference more than Anthropic / OpenAI APIs
- Physical touch builds trust I guess: we have started from a handshake and finished with a hug which even to me felt quite friendly and full of trust

**things to have ready**
- security page
	- data retention -> ZDR in bigger companies + cloud inference
	- data encryption + services used
- demo ready
- pens (black and red are best) and paper to brake the third wall
- informal settings (maybe)
- The graph to show them

## Knowledge
### What is the problem
- They hire new ppl and cannot trust them for a year
- New ppl are extremely **risk-averse** compared to Tess
- **Her Pain:** 
	- Tess has 6 hours of meetings daily + cannot do the work she wants for 4 years
		- constant ppl mentorship
		- ppl needing a client call over-sitting in case they fail
		- back-and-force with client's IT team on what they need
	- takes too long to teach business ppl how to use excel task completion prediction excel sheet
- **Company Pain:**
	- Bookkeeping
		- Company asks ppl to write down their tacit knowledge
			- senior ppl may do this
			- junior ppl do not
	- ==Unknown Unknowns== -- ppl do not now what to even search for!
	- Demos that are made to teach ppl are bad --> not useful to teach
	- Saved data is messy
- **Attempts**
	- Direct Mentorship
		- Year 1: did weekend webinars for her team and the company
		- Year 2-3: sit side-by-side teaching ppl how to click buttons + made checklists
		- Year 4: taught others and now they own folders in SharePoint library
		- **Problem:** not really scalable, takes lots of time and huge amount of money
	- They also have video guides on
		- Internal hour-billing
		- Software configuration
		- What to say to clients
		- Culture - how to write emails
		- How, Why, Where to find
		- PM skills, Time management skills
		- **Problem:** ppl do not want to watch hour-long content, too much for them
	- They also have lots of word documents
		- **Problem:** FRICTION -> ppl do not use search bar, ppl complain they cannot find things
	- Tess tried to configure Copilot no code, but failed in giving structured outputs for her template
- **What they have now**
	- Tess made checklists
	- Tess taught a group of ppl who own folders now
		- 10 senior ppl who spend 16h/3 month (should spend 24h/month)
			- total of 10 * 24 * 35 = 8400 CAD of time per month to do the bookkeeping
	- They use ==SharePoint== as the source of truth

**side pains**
- business grads do not know how to generalize their skills of communication
- new hires cannot uncover client's problems because they do not know what to ask (==unknown unknowns!==)
- new hires are afraid of meetings where they may not know how smth specific is solved

**hidden pains**
- moving to a new ecosystem --> like she was really happy to use the DB they have in place
- getting legal docs --> to hire me they needed some insurance 
- not every software they use has an open API (ex SAP does not!)

**side facts**
- they use the whole microsoft suite 
	- teams
	- copilot
	- sharepoint
	- some random graph DB
- they have 300 ppl 
- Tess mentioned that my big bet would be to sell to small "innovative" teams in the larger conservative corporations 
	- because these teams exist to experiment with such tooling

**what we have concluded on**
- She already mentioned me to the team in US and they have shortage of ML / Tech ppl
- She is a local team lead, so she owns a lot of leadership
- She got trust from the budget holders
- It seemed to me that she sees me as an expert and trusts, so it may go well
- Tess mentioned that she will try to play around the sandboxes they have in place to see if anything could be done (idk if she wants to implement it by herself or not, but in any case something feels to me that if she tries she won't have time to complete it)
- also they contract ppl when there is a shortage of resources internally
	- like some other python devs
- We wrapped up on these action points
	- She will try out the sandboxes for the graph DBs she has access to
	- She will talk about hiring me to ppl in US
- ==**HER INTEREST:** she want to have a proof of concept for the better usage of AI! and this is the way she is thinking about making it happen!==
- Tess is not the budget owner, but the strongest champion I could have had

**good signs**
- she has found time for me in 6h meetings calendar +30 mins before and after
- she proposed next steps

**ideas for the follow-up**
- make it easier for her to push the idea with the 

## Follow-up
**Today:** send a short warm text
**Tmrw:** draft up a security doc
**Later during the week:** ask how the thing with her DB works


really enjoyed today, thanks for making room around that calendar haha!

quick gut-check that I got your world right: ~a year before new hires can be trusted with client work; knowledge lives in your (now also owners) heads or docs/vids no one can find, and the "people don't even know what to search for" is most problematic where the usual FAQ db fails

let me know how the sandbox poking goes, happy to jump in if you hit a wall, by the way -- which db provider do you use?


I can clearly see that there are AI-KMS for enteprise form hte enterprise
- For example there are KMS's from Amazon / Google
	- Amazon Q + some wrappers
		- https://aws.amazon.com/marketplace/pp/prodview-2spt7wntxondy
		- https://aws.amazon.com/marketplace/pp/prodview-nme7nt5v7pdco
		- Pros
			- Context-aware
			- Self-improving (?)
			- Versioning metadata
			- Custom portals
		- Cons
			- 5 week deployment
			- docs only
- Zen - YC baked
	- Temporal graph agent memory
	- open sourced part of the product -> Graphiti
2. """

chunks = chunk_doc(doc)
print(f"{len(chunks)} chunks, sizes: {[len(c) for c in chunks]}")

11 chunks, sizes: [186, 851, 337, 1415, 1951, 270, 1945, 835, 1037, 821, 525]


In [13]:
# per-chunk: extract typed entities and embed each span using only that chunk as context
vectors: dict[str, list[float]] = {}
entity_types: dict[str, EntityType] = {}

for i, chunk in enumerate(chunks):
    for entity in extract_entities(chunk):
        if entity.text in vectors:
            continue
        try:
            start = chunk.index(entity.text)
            vectors[entity.text] = embed_span(chunk, start, start + len(entity.text))
            entity_types[entity.text] = entity.type
        except ValueError:
            print(f"  skipping '{entity.text}' — not found verbatim in chunk {i}")

print(f"{len(vectors)} entities embedded")
for text, etype in entity_types.items():
    print(f"  [{etype.value:<10}] {text}")

  skipping 'Slack' — not found verbatim in chunk 2
57 entities embedded
  [ROLE      ] CTO's
  [ROLE      ] COO's
  [CONCEPT   ] late Seed
  [CONCEPT   ] Series A
  [ORG       ] B2B SaaS startups
  [CONCEPT   ] Champions + Buyers
  [ROLE      ] CTO
  [ROLE      ] VP of Engineering
  [ROLE      ] COO
  [ROLE      ] Head of Ops
  [ROLE      ] CEO
  [ROLE      ] Senior engineer
  [ROLE      ] tech lead
  [CONCEPT   ] bottom-up demand signal
  [CONCEPT   ] Fintech
  [CONCEPT   ] biotech
  [CONCEPT   ] B2B SaaS
  [CONCEPT   ] first painful departure
  [ORG       ] team
  [CONCEPT   ] post-incident relief
  [TOOL      ] GraphRAG
  [TOOL      ] FAQ RAG
  [ROLE      ] security HR
  [ROLE      ] international HR
  [CONCEPT   ] Mom test
  [TOOL      ] Zero Data Retention
  [CONCEPT   ] RAG
  [PERSON    ] Tess
  [ORG       ] Anthropic
  [ORG       ] OpenAI
  [ORG       ] ZDR
  [CONCEPT   ] cloud inference
  [TOOL      ] black
  [TOOL      ] red
  [ORG       ] SharePoint
  [ORG       ] SAP
  [ORG 

In [ ]:
def string_overlap(a: str, b: str) -> float:
    """Word-level Jaccard; returns 1.0 if one string contains the other."""
    a_l, b_l = a.lower(), b.lower()
    if a_l in b_l or b_l in a_l:
        return 1.0
    words_a, words_b = set(a_l.split()), set(b_l.split())
    if not words_a or not words_b:
        return 0.0
    return len(words_a & words_b) / len(words_a | words_b)

def deduplicate(
    vectors: dict[str, list[float]],
    entity_types: dict[str, EntityType],
    str_threshold: float = 0.5,
    sim_threshold: float = 0.85,
) -> tuple[dict[str, list[float]], dict[str, EntityType]]:
    """Merge entities that share type + string overlap + embedding similarity."""
    entities = list(vectors.keys())
    parent = {e: e for e in entities}

    def find(x: str) -> str:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x: str, y: str) -> None:
        rx, ry = find(x), find(y)
        if rx != ry:
            # keep the longer name as canonical
            parent[ry if len(rx) >= len(ry) else rx] = rx if len(rx) >= len(ry) else ry

    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):
            a, b = entities[i], entities[j]
            if entity_types.get(a) != entity_types.get(b):
                continue
            if string_overlap(a, b) < str_threshold:
                continue
            if similarity(vectors[a], vectors[b]) < sim_threshold:
                continue
            union(a, b)

    groups: dict[str, list[str]] = defaultdict(list)
    for e in entities:
        groups[find(e)].append(e)

    dedup_vectors: dict[str, list[float]] = {}
    dedup_types: dict[str, EntityType] = {}
    for canonical, members in groups.items():
        dedup_vectors[canonical] = np.mean([vectors[m] for m in members], axis=0).tolist()
        dedup_types[canonical] = entity_types[canonical]
        if len(members) > 1:
            print(f"merged {members} → '{canonical}'")

    print(f"\n{len(vectors)} → {len(dedup_vectors)} entities after dedup")
    return dedup_vectors, dedup_types


dedup_vectors, dedup_types = deduplicate(vectors, entity_types)

## Correlation Clustering (Pivot Algorithm)

Groups entities without specifying k. Minimizes disagreements: positive edges (sim > threshold) across clusters + negative edges (sim ≤ threshold) within clusters.

In [ ]:
import random

def correlation_clustering(vectors: dict[str, list[float]], threshold: float = 0.6) -> list[list[str]]:
    """Pivot algorithm: 3-approximation, O(n²), no k needed."""
    remaining = list(vectors.keys())
    random.shuffle(remaining)
    clusters = []

    while remaining:
        pivot = remaining.pop(0)
        cluster = [pivot]
        new_remaining = []
        for node in remaining:
            if similarity(vectors[pivot], vectors[node]) > threshold:
                cluster.append(node)
            else:
                new_remaining.append(node)
        clusters.append(cluster)
        remaining = new_remaining

    return clusters

In [ ]:
from collections import defaultdict

def consensus_clustering(vectors: dict[str, list[float]], threshold: float = 0.7, n_runs: int = 50) -> list[list[str]]:
    entities = list(vectors.keys())
    n = len(entities)
    idx = {e: i for i, e in enumerate(entities)}

    # count how often each pair lands in the same cluster across runs
    co_occurrence = defaultdict(int)
    for _ in range(n_runs):
        for cluster in correlation_clustering(vectors, threshold):
            for i in range(len(cluster)):
                for j in range(i + 1, len(cluster)):
                    a, b = sorted([cluster[i], cluster[j]])
                    co_occurrence[(a, b)] += 1

    # normalize to [0, 1]
    co_sim = {pair: count / n_runs for pair, count in co_occurrence.items()}

    # build co-occurrence vectors per entity and run one final clustering pass
    co_vectors = {}
    for e in entities:
        vec = []
        for other in entities:
            if e == other:
                vec.append(1.0)
            else:
                pair = tuple(sorted([e, other]))
                vec.append(co_sim.get(pair, 0.0))
        co_vectors[e] = vec

    return correlation_clustering(co_vectors, threshold=0.5)


clusters = consensus_clustering(vectors, threshold=0.6, n_runs=120)
for i, cluster in enumerate(clusters):
    print(f"Cluster {i + 1}: {cluster}")

In [ ]:
def assign_to_cluster(
    entity_vec: list[float],
    clusters: list[list[str]],
    vectors: dict[str, list[float]],
    threshold: float = 0.35,
) -> tuple[int, float] | tuple[None, None]:
    """Return (cluster_index, score) for the best matching cluster, or (None, None)."""
    best_idx, best_score = None, threshold
    for i, cluster in enumerate(clusters):
        members = [vectors[m] for m in cluster if m in vectors]
        if not members:
            continue
        centroid = np.mean(members, axis=0)
        score = similarity(entity_vec, centroid.tolist())
        if score > best_score:
            best_score, best_idx = score, i
    return best_idx, best_score if best_idx is not None else (None, None)

def process_new_doc(new_doc: str, clusters: list[list[str]], vectors: dict[str, list[float]]) -> None:
    """Extract entities from a new doc and assign each to an existing cluster."""
    new_chunks = chunk_doc(new_doc)
    print(f"new doc: {len(new_chunks)} chunks\n")

    for chunk in new_chunks:
        for entity in extract_entities(chunk):
            try:
                start = chunk.index(entity.text)
                vec = embed_span(chunk, start, start + len(entity.text))
                idx, score = assign_to_cluster(vec, clusters, vectors)
                if idx is not None:
                    print(f"  [{entity.type.value:<10}] '{entity.text}' → Cluster {idx + 1} {clusters[idx]} (score={score:.3f})")
                else:
                    print(f"  [{entity.type.value:<10}] '{entity.text}' → NEW (no cluster match)")
            except ValueError:
                pass


# run consensus clustering on deduped vectors first, then assign from a new doc
clusters = consensus_clustering(dedup_vectors, threshold=0.5, n_runs=50)

new_doc = """
Met with Sarah Chen, CPO at Notion, to discuss their enterprise knowledge management challenges.
They use Confluence and Slack heavily but struggle with information retrieval.
Sarah mentioned that their ML team is experimenting with LangChain and vector databases like Pinecone.
OpenAI API costs are a concern at their scale.
"""

process_new_doc(new_doc, hdb_clusters, dedup_vectors)

In [ ]:
def mahalanobis_distance(
    x: np.ndarray,
    members: list[list[float]],
    reg: float = 1e-3,
) -> float:
    """
    Diagonal-covariance Mahalanobis distance.
    Full covariance needs hundreds of samples; diagonal (per-dim variance)
    is stable with small clusters.
    """
    X = np.array(members)
    mu = X.mean(axis=0)
    if len(X) < 2:
        return float(np.linalg.norm(x - mu))  # fallback: euclidean
    var = X.var(axis=0) + reg
    return float(np.sqrt(((x - mu) ** 2 / var).sum()))


def assign_to_cluster_mahalanobis(
    entity_vec: list[float],
    clusters: list[list[str]],
    vectors: dict[str, list[float]],
    distance_threshold: float = 50.0,
) -> tuple[int, float] | tuple[None, None]:
    x = np.array(entity_vec)
    best_idx, best_dist = None, distance_threshold
    for i, cluster in enumerate(clusters):
        members = [vectors[m] for m in cluster if m in vectors]
        if not members:
            continue
        dist = mahalanobis_distance(x, members)
        if dist < best_dist:
            best_dist, best_idx = dist, i
    return (best_idx, best_dist) if best_idx is not None else (None, None)


# side-by-side comparison
print("=== Centroid (cosine sim) ===")
for chunk in chunk_doc(new_doc):
    for entity in extract_entities(chunk):
        try:
            start = chunk.index(entity.text)
            vec = embed_span(chunk, start, start + len(entity.text))
            idx, score = assign_to_cluster(vec, hdb_clusters, dedup_vectors)
            label = f"Cluster {idx+1} ({score:.3f})" if idx is not None else "NEW"
            print(f"  [{entity.type.value:<10}] {entity.text:<20} → {label}")
        except ValueError:
            pass

print("\n=== Gaussian / Mahalanobis ===")
for chunk in chunk_doc(new_doc):
    for entity in extract_entities(chunk):
        try:
            start = chunk.index(entity.text)
            vec = embed_span(chunk, start, start + len(entity.text))
            idx, dist = assign_to_cluster_mahalanobis(vec, hdb_clusters, dedup_vectors)
            label = f"Cluster {idx+1} (dist={dist:.1f})" if idx is not None else "NEW"
            print(f"  [{entity.type.value:<10}] {entity.text:<20} → {label}")
        except ValueError:
            pass

## HDBSCAN Clustering

Density-based hierarchical clustering. No k needed, no threshold to tune — just `min_cluster_size`. Automatically marks low-density outliers as noise (-1).

In [ ]:
from sklearn.cluster import HDBSCAN

def hdbscan_clustering(
    vectors: dict[str, list[float]],
    min_cluster_size: int = 2,
    min_samples: int = 1,
    metric: str = "cosine",
) -> tuple[list[list[str]], list[str]]:
    """Returns (clusters, noise). Noise entities didn't fit any cluster."""
    entities = list(vectors.keys())
    X = np.array([vectors[e] for e in entities])

    labels = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=metric,
    ).fit_predict(X)

    clusters: dict[int, list[str]] = defaultdict(list)
    noise: list[str] = []
    for entity, label in zip(entities, labels):
        if label == -1:
            noise.append(entity)
        else:
            clusters[label].append(entity)

    return list(clusters.values()), noise


hdb_clusters, noise = hdbscan_clustering(dedup_vectors, min_cluster_size=2, min_samples=1)

print(f"{len(hdb_clusters)} clusters, {len(noise)} noise points\n")
for i, cluster in enumerate(hdb_clusters):
    print(f"Cluster {i + 1}: {cluster}")
if noise:
    print(f"\nNoise (outliers): {noise}")

In [ ]:
def process_new_doc_hdbscan(
    new_doc: str,
    hdb_clusters: list[list[str]],
    vectors: dict[str, list[float]],
    threshold: float = 0.35,
) -> tuple[list[list[str]], dict[str, list[float]]]:
    n_original = len(hdb_clusters)

    # freeze centroids before processing — never updated mid-doc
    centroids = [
        np.mean([vectors[m] for m in hdb_clusters[i] if m in vectors], axis=0)
        for i in range(n_original)
    ]

    for chunk in chunk_doc(new_doc):
        for entity in extract_entities(chunk):
            try:
                start = chunk.index(entity.text)
                vec = embed_span(chunk, start, start + len(entity.text))
            except ValueError:
                continue

            best_idx, best_score = None, threshold
            for i, centroid in enumerate(centroids):
                score = similarity(vec, centroid.tolist())
                if score > best_score:
                    best_score, best_idx = score, i

            vectors[entity.text] = vec

            if best_idx is not None:
                hdb_clusters[best_idx].append(entity.text)
                print(f"  [{entity.type.value:<10}] '{entity.text}' → Cluster {best_idx + 1} {hdb_clusters[best_idx][:-1]} ({best_score:.3f})")
            else:
                hdb_clusters.append([entity.text])
                print(f"  [{entity.type.value:<10}] '{entity.text}' → NEW cluster {len(hdb_clusters)}")

    return hdb_clusters, vectors


new_doc = """
Met with Sarah Chen, CPO at Notion, to discuss their enterprise knowledge management challenges.
They use Confluence and Slack heavily but struggle with information retrieval.
Sarah mentioned that their ML team is experimenting with LangChain and vector databases like Pinecone.
OpenAI API costs are a concern at their scale.
"""

hdb_clusters, dedup_vectors = process_new_doc_hdbscan(new_doc, hdb_clusters, dedup_vectors)
print(f"\n{len(hdb_clusters)} clusters after update")

In [ ]:

Cluster 1: ["CTO's", "COO's"]
Cluster 2: ['B2B SaaS', 'startups']
Cluster 3: ['Champions + Buyers', 'bottom-up demand signal']
Cluster 4: ['CTO', 'COO', 'Head of Ops', 'CEO', 'co-founder']
Cluster 5: ['VP of Engineering', 'Senior engineer']
Cluster 6: ['Screenshot 2026-05-14 at 12.04.09.png', 'Today', 'Tmrw']
Cluster 7: ['GraphRAG', 'FAQ RAG', 'RAG', 'FAQ', 'sandbox']
Cluster 8: ['Mom test', 'Tess']
Cluster 9: ['Anthropic', 'OpenAI']
Cluster 10: ['ZDR', 'cloud']
Cluster 11: ['SharePoint', 'DB', 'SAP', 'microsoft suite', 'teams', 'copilot', 'sharepoint']
Cluster 12: ['ML', 'Tech', 'AI']
Cluster 13: ['AI-KMS', 'Zen', 'YC', 'Graphiti']
Cluster 14: ['Amazon', 'Google', 'Amazon Q']
Cluster 15: ['Sarah Chen', 'CPO', 'Notion']
Cluster 16: ['Slack', 'LangChain', 'Pinecone', 'OpenAI API']

Noise (outliers): ['Seed', 'Series A', 'tech lead', 'HR', 'Zero Data Retention', 'CAD', 'US', 'Confluence']

## Asymmetric Retrieval with E5

E5 uses **asymmetric encoding**: questions get a `query:` prefix, facts/passages get a `passage:` prefix. The model is trained on this asymmetry — omitting either prefix silently degrades scores.

- `intfloat/e5-base-v2` — 768-dim, ~440MB, good default
- `intfloat/e5-large-v2` — 1024-dim, ~1.3GB, better accuracy
- `intfloat/multilingual-e5-large` — use if any content is non-English

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# swap to "intfloat/e5-large-v2" (1024-dim) or "intfloat/multilingual-e5-large" as needed
E5_MODEL = "intfloat/e5-base-v2"

e5_tokenizer = AutoTokenizer.from_pretrained(E5_MODEL)
e5_model = AutoModel.from_pretrained(E5_MODEL)
e5_model.eval()
print(f"loaded {E5_MODEL}")

In [ ]:
def _e5_encode(texts: list[str]) -> np.ndarray:
    """Encode a batch of already-prefixed texts. Returns L2-normalised vectors."""
    enc = e5_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )
    with torch.no_grad():
        out = e5_model(**enc)
    # attention-mask weighted mean pool (standard E5 recipe)
    mask = enc["attention_mask"].unsqueeze(-1).float()
    pooled = (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
    return F.normalize(pooled, p=2, dim=1).numpy()


def encode_query(text: str) -> np.ndarray:
    """Encode a natural-language question. Always adds 'query:' prefix."""
    return _e5_encode([f"query: {text}"])[0]


def encode_passage(text: str) -> np.ndarray:
    """Encode a fact / node text. Always adds 'passage:' prefix."""
    return _e5_encode([f"passage: {text}"])[0]


def encode_passages(texts: list[str]) -> np.ndarray:
    """Batch-encode multiple passages. Each gets 'passage:' prefix."""
    return _e5_encode([f"passage: {t}" for t in texts])

In [ ]:
def retrieve(
    question: str,
    passages: list[str],
    passage_vecs: np.ndarray,
    top_k: int = 5,
) -> list[tuple[float, str]]:
    """Return top_k (score, passage) pairs for a question."""
    q_vec = encode_query(question)
    scores = passage_vecs @ q_vec  # dot product on L2-normed vecs = cosine sim
    top_indices = scores.argsort()[::-1][:top_k]
    return [(float(scores[i]), passages[i]) for i in top_indices]

In [ ]:
# ── Build two passage indexes from the data already in the notebook ──────────

# Index 1: individual entity names (leaf-level facts)
entity_passages = list(dedup_vectors.keys())
entity_vecs = encode_passages(entity_passages)

# Index 2: cluster summaries — members joined as a phrase
cluster_passages = [", ".join(c) for c in hdb_clusters]
cluster_vecs = encode_passages(cluster_passages)

print(f"entity index:  {len(entity_passages)} passages")
print(f"cluster index: {len(cluster_passages)} passages")

In [ ]:
# ── Test: question-fact retrieval ─────────────────────────────────────────────

questions = [
    "Who are the best people to talk to at a startup about knowledge management?",
    "What tools does the company use for internal knowledge?",
    "What makes GraphRAG better than traditional FAQ search?",
    "What was the main pain Tess described?",
    "Which AI companies are competitors?",
]

for q in questions:
    print(f"Q: {q}")
    print("  — entity hits:")
    for score, passage in retrieve(q, entity_passages, entity_vecs, top_k=4):
        print(f"     {score:.3f}  {passage}")
    print("  — cluster hits:")
    for score, passage in retrieve(q, cluster_passages, cluster_vecs, top_k=2):
        print(f"     {score:.3f}  [{passage}]")
    print()

In [435]:
# ── Prefix sanity check: shows score drop when prefix is omitted ──────────────
# This is the most common silent bug with E5 — everything looks reasonable
# but scores are systematically lower and rankings drift.

q_text = "What tools does the company use for internal knowledge?"
passage_text = "SharePoint, DB, SAP, microsoft suite, teams, copilot, sharepoint"

q_with    = _e5_encode([f"query: {q_text}"])[0]
q_without = _e5_encode([q_text])[0]
p_with    = _e5_encode([f"passage: {passage_text}"])[0]
p_without = _e5_encode([passage_text])[0]

print("query: prefix + passage: prefix  →", f"{float(q_with @ p_with):.4f}   ← correct")
print("no query prefix + passage: prefix →", f"{float(q_without @ p_with):.4f}")
print("query: prefix + no passage prefix →", f"{float(q_with @ p_without):.4f}")
print("no prefixes at all                →", f"{float(q_without @ p_without):.4f}")

query: prefix + passage: prefix  → 0.7839   ← correct
no query prefix + passage: prefix → 0.7865
query: prefix + no passage prefix → 0.7779
no prefixes at all                → 0.7884


In [28]:
# ── Prefix effect on long passages (the real use case) ───────────────────────
# Short entity names ≈ symmetric → prefix barely moves the score.
# Long document chunks ≈ asymmetric → prefix alignment is what E5 is trained for.

q_text = "What tools does the company use for internal knowledge?"

# pick the chunk that actually mentions SharePoint / tools
long_passage = next(c for c in chunks if "SharePoint" in c)
print(f"passage length: {len(long_passage)} chars\n{long_passage[:120]}...\n")

q_with    = _e5_encode([f"query: {q_text}"])[0]
q_without = _e5_encode([q_text])[0]
p_with    = _e5_encode([f"passage: {long_passage}"])[0]
p_without = _e5_encode([long_passage])[0]

print("query: + passage:  →", f"{float(q_with @ p_with):.4f}   ← correct")
print("no query + passage →", f"{float(q_without @ p_with):.4f}")
print("query + no passage →", f"{float(q_with @ p_without):.4f}")
print("no prefixes        →", f"{float(q_without @ p_without):.4f}")
print(f"\ndelta (correct vs no prefixes): {float(q_with @ p_with) - float(q_without @ p_without):+.4f}")

passage length: 1945 chars
## Knowledge
### What is the problem
- They hire new ppl and cannot trust them for a year
- New ppl are extremely **risk...



NameError: name '_e5_encode' is not defined